# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
import os
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from agent import Agent

load_dotenv()

True

In [2]:
ECOHOME_SYSTEM_PROMPT = """You are the EcoHome Energy Advisor, an AI assistant specializing in smart-home energy optimization. You help homeowners reduce electricity costs and environmental impact through personalized, data-driven recommendations.

Your Role:
You analyze energy usage patterns, weather forecasts, electricity pricing, and solar generation data to provide actionable recommendations for optimal device scheduling and energy management.

Step-by-Step Workflow:
1. Understand the user's question and identify which devices or systems are involved.
2. Gather relevant data using your available tools:
   - get_weather_forecast: for solar irradiance predictions and temperature
   - get_electricity_prices: for time-of-use pricing schedules
   - query_energy_usage: for historical consumption patterns
   - query_solar_generation: for past solar production data
   - get_recent_energy_summary: for a quick snapshot of recent usage
   - search_energy_tips: for energy-saving best practices from the knowledge base
   - calculate_energy_savings: to compute potential cost savings
3. Analyze the data to identify optimal timing and settings.
4. Provide a clear, specific recommendation with exact times, estimated savings, and supporting reasoning.
5. Reference relevant energy-saving tips from the knowledge base when available.

Key Capabilities:
- Multi-device optimization: EVs, HVAC, appliances, solar systems
- Weather-aware scheduling: align energy-intensive tasks with solar generation
- Cost optimization: leverage off-peak pricing windows
- Historical analysis: identify patterns in usage data
- Savings calculations: quantify the benefit of each recommendation

Recommendation Guidelines:
- Always provide specific, actionable advice (exact times, temperatures, settings)
- Include estimated cost savings when possible
- Consider both cost and environmental impact
- Reference relevant energy-saving tips when available
- If data is insufficient, state what additional information would help
- Provide clear reasoning for each recommendation

Example Questions:
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"
- "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?"
- "Suggest three ways I can reduce energy use based on my usage history."
- "How much can I save by running my dishwasher during off-peak hours?"
- "What's the best time to run my pool pump this week based on the weather forecast?"
- "Should I install battery storage to maximize my solar panel investment?"
- "How can I optimize my HVAC schedule for tomorrow's weather?"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric car tomorrow (October 4, 2023), here are the optimal charging times based on the weather forecast and electricity pricing:

### Optimal Charging Time:
- **Charge your electric car between 12:00 PM and 2:00 PM.**

### Reasoning:
1. **Solar Power Generation:**
   - The solar irradiance is expected to peak around **12:00 PM** with a value of **784.3 W/m²**, which means solar generation will be at its highest during this time.
   - The solar power will start to decline after 1:00 PM, so charging during this window will allow you to utilize the maximum solar energy.

2. **Electricity Pricing:**
   - The electricity rates during this time are as follows:
     - **12:00 PM:** $0.1781 per kWh (on-peak)
     - **1:00 PM:** $0.1706 per kWh (on-peak)
   - While these rates are on-peak, they are still lower than the rates during the later afternoon and evening hours, which can go up to $0.1856 per kWh.

### Summary:
- **Best Ti

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump() if hasattr(msg, 'model_dump') else msg
    if isinstance(obj, dict) and obj.get("tool_call_id"):
        print("-", msg.name if hasattr(msg, 'name') else obj.get('name', 'unknown'))

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [7]:
# 12 comprehensive test cases covering EV, thermostat, appliances, solar, cost savings
test_cases = [
    {
        "id": "ev_charging_1",
        "category": "EV",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should recommend charging times that align with off-peak electricity rates and/or high solar generation periods, with specific time windows and cost savings.",
    },
    {
        "id": "ev_charging_2",
        "category": "EV",
        "question": "How much would I save per month if I charge my EV at night (off-peak) instead of during the day (on-peak) if I use 300 kWh per month?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should calculate the savings between off-peak and on-peak charging rates and provide a monthly dollar amount.",
    },
    {
        "id": "thermostat_1",
        "category": "thermostat",
        "question": "What temperature should I set my thermostat this afternoon to save energy based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should recommend a specific temperature setting based on outdoor temperature and electricity pricing, with reference to energy-saving tips.",
    },
    {
        "id": "thermostat_2",
        "category": "thermostat",
        "question": "How can I reduce my heating costs this winter using my smart thermostat?",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "The response should provide actionable thermostat scheduling strategies for winter energy savings and reference relevant tips.",
    },
    {
        "id": "appliance_1",
        "category": "appliances",
        "question": "When should I run my dishwasher to minimize electricity cost based on today's pricing?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should identify the cheapest time-of-use period for running the dishwasher and suggest cost savings.",
    },
    {
        "id": "appliance_2",
        "category": "appliances",
        "question": "What is the cheapest time to do laundry this week given the electricity price schedule?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should recommend specific hours or days with the lowest electricity rates for running washing machines and dryers.",
    },
    {
        "id": "solar_1",
        "category": "solar",
        "question": "How can I maximize the use of my solar panels today based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "search_energy_tips"],
        "expected_response": "The response should analyze solar generation potential and recommend scheduling high-consumption devices during peak solar hours.",
    },
    {
        "id": "solar_2",
        "category": "solar",
        "question": "Should I sell excess solar power back to the grid or store it in my battery system?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should compare the economics of selling to the grid versus storing, based on pricing and energy storage best practices.",
    },
    {
        "id": "cost_savings_1",
        "category": "cost_savings",
        "question": "What are my biggest energy expenses based on my recent usage history and how can I reduce them?",
        "expected_tools": ["query_energy_usage", "get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "The response should identify the highest consumption devices and provide specific reduction strategies with savings estimates.",
    },
    {
        "id": "cost_savings_2",
        "category": "cost_savings",
        "question": "Calculate my potential monthly savings if I shift all my appliance usage from on-peak to off-peak hours assuming I use 150 kWh for appliances per month.",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should compute the rate differential and provide a specific monthly savings amount.",
    },
    {
        "id": "general_1",
        "category": "general",
        "question": "Give me a summary of my energy usage and solar generation for the past week.",
        "expected_tools": ["query_energy_usage", "query_solar_generation", "get_recent_energy_summary"],
        "expected_response": "The response should provide a comprehensive summary of total consumption, generation, and cost.",
    },
    {
        "id": "hvac_specific",
        "category": "thermostat",
        "question": "Based on tomorrow's weather forecast, when should I run my HVAC system most efficiently?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should recommend specific HVAC scheduling aligned with weather patterns and electricity pricing.",
    },
]

categories = set(tc['category'] for tc in test_cases)
print(f"Total test cases: {len(test_cases)}")
print(f"Categories covered: {categories}")

if len(test_cases) < 10:
    raise ValueError(f"You MUST have at least 10 test cases, but only {len(test_cases)} defined")

Total test cases: 12
Categories covered: {'appliances', 'solar', 'thermostat', 'cost_savings', 'EV', 'general'}


## 3. Run Agent Tests

In [8]:
CONTEXT = "Location: San Francisco, CA"

In [9]:
print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']} ({test_case['category']})")
    print(f"Question: {test_case['question']}")
    print("-" * 50)

    try:
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )

        result = {
            'test_id': test_case['id'],
            'category': test_case['category'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)

    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'category': test_case['category'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")

=== Running Agent Tests ===

Test 1: ev_charging_1 (EV)
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2 (EV)
Question: How much would I save per month if I charge my EV at night (off-peak) instead of during the day (on-peak) if I use 300 kWh per month?
--------------------------------------------------

Test 3: thermostat_1 (thermostat)
Question: What temperature should I set my thermostat this afternoon to save energy based on the weather forecast?
--------------------------------------------------

Test 4: thermostat_2 (thermostat)
Question: How can I reduce my heating costs this winter using my smart thermostat?
--------------------------------------------------

Test 5: appliance_1 (appliances)
Question: When should I run my dishwasher to minimize electricity cost based on today's pricing?
--------------------------------------------------

Test 6: appliance_

In [10]:
test_results

[{'test_id': 'ev_charging_1',
  'category': 'EV',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='98f6ed27-6029-4cb5-8446-407d15e67f14'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='34619b96-f205-42c5-9b5d-d747857bf68b'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 1158, 'total_tokens': 1219, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 

## 4. Evaluate Responses

In [11]:
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response using LLM scoring.

    Args:
        question (str): The original user question
        final_response (str): The agent's final response text
        expected_response (str): Description of what a good response should contain

    Returns:
        dict: Scores for ACCURACY, RELEVANCE, COMPLETENESS, USEFULNESS and feedback
    """
    try:
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.0,
            base_url="https://openai.vocareum.com/v1",
            api_key=os.getenv("VOCAREUM_API_KEY")
        )

        scoring_prompt = f"""You are an evaluation judge. Score the following response on four dimensions from 1 (worst) to 10 (best). Return ONLY a JSON object with no other text.

Question: {question}

Expected: {expected_response}

Actual Response: {final_response}

{{
  "ACCURACY": <int 1-10>,
  "RELEVANCE": <int 1-10>,
  "COMPLETENESS": <int 1-10>,
  "USEFULNESS": <int 1-10>,
  "feedback": "<brief feedback explaining the scores>"
}}"""

        result = llm.invoke(scoring_prompt)
        content = result.content.strip()

        if content.startswith("```"):
            content = content.split("\n", 1)[1]
            content = content.rsplit("\n```", 1)[0] if "```" in content else content

        scores = json.loads(content)

        return {
            "ACCURACY": scores.get("ACCURACY", 5),
            "RELEVANCE": scores.get("RELEVANCE", 5),
            "COMPLETENESS": scores.get("COMPLETENESS", 5),
            "USEFULNESS": scores.get("USEFULNESS", 5),
            "feedback": scores.get("feedback", "No feedback provided")
        }
    except Exception as e:
        return {
            "ACCURACY": 5,
            "RELEVANCE": 5,
            "COMPLETENESS": 5,
            "USEFULNESS": 5,
            "feedback": f"Evaluation scoring error: {str(e)}. Default scores assigned."
        }

In [12]:
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used.

    Args:
        messages (list): List of message objects from agent response
        expected_tools (list): List of expected tool names

    Returns:
        dict: Tool Appropriateness, Tool Completeness scores and feedback
    """
    tools_used = set()
    if isinstance(messages, list):
        for msg in messages:
            if hasattr(msg, 'model_dump'):
                obj = msg.model_dump()
            elif isinstance(msg, dict):
                obj = msg
            else:
                continue

            if obj.get("tool_call_id"):
                name = obj.get("name", "") or getattr(msg, 'name', '')
                if name:
                    tools_used.add(name)

    expected_set = set(expected_tools)

    if len(tools_used) == 0:
        return {
            "Tool Appropriateness": 0,
            "Tool Completeness": 0,
            "tools_used": list(tools_used),
            "expected_tools": expected_tools,
            "feedback": "No tools were used in the response."
        }

    correct_tools = tools_used & expected_set
    unnecessary_tools = tools_used - expected_set

    appropriateness = round(len(correct_tools) / len(tools_used) * 10, 1) if tools_used else 0
    completeness = round(len(correct_tools) / len(expected_set) * 10, 1) if expected_set else 10

    feedback_parts = []
    if correct_tools:
        feedback_parts.append(f"Correctly used: {', '.join(sorted(correct_tools))}")
    if unnecessary_tools:
        feedback_parts.append(f"Unnecessary tools used: {', '.join(sorted(unnecessary_tools))}")
    missing = expected_set - tools_used
    if missing:
        feedback_parts.append(f"Missing expected tools: {', '.join(sorted(missing))}")

    return {
        "Tool Appropriateness": appropriateness,
        "Tool Completeness": completeness,
        "tools_used": list(tools_used),
        "expected_tools": expected_tools,
        "feedback": "; ".join(feedback_parts) if feedback_parts else "Tool usage matches expectations."
    }

In [13]:
def generate_evaluation_report():
    """Generate a comprehensive evaluation report based on test results.

    Evaluates response quality and tool usage for each test case,
    aggregates scores, identifies strengths and weaknesses,
    and saves the report to /reports/ as a markdown file.
    """
    if not test_results:
        print("No test results to evaluate. Run the agent tests first.")
        return

    all_evaluations = []

    for result in test_results:
        question = result['question']
        expected_response = result['expected_response']
        expected_tools = result['expected_tools']

        if isinstance(result['response'], dict) and 'messages' in result['response']:
            messages = result['response']['messages']
            final_content = ""
            for msg in messages:
                if hasattr(msg, 'content') and msg.content:
                    final_content = msg.content
                elif isinstance(msg, dict) and msg.get('content'):
                    final_content = msg['content']
        elif isinstance(result['response'], str):
            final_content = result['response']
            messages = []
        else:
            final_content = str(result['response'])
            messages = []

        response_eval = evaluate_response(question, final_content, expected_response)
        tool_eval = evaluate_tool_usage(messages, expected_tools)

        all_evaluations.append({
            'test_id': result['test_id'],
            'category': result.get('category', 'general'),
            'question': question,
            'response_eval': response_eval,
            'tool_eval': tool_eval
        })

    # Aggregate scores
    metrics = ['ACCURACY', 'RELEVANCE', 'COMPLETENESS', 'USEFULNESS']
    tool_metrics = ['Tool Appropriateness', 'Tool Completeness']

    agg = {m: {'scores': [], 'all_feedback': []} for m in metrics + tool_metrics}

    for ev in all_evaluations:
        for m in metrics:
            agg[m]['scores'].append(ev['response_eval'][m])
            agg[m]['all_feedback'].append(ev['response_eval']['feedback'])
        for m in tool_metrics:
            agg[m]['scores'].append(ev['tool_eval'][m])
            agg[m]['all_feedback'].append(ev['tool_eval']['feedback'])

    # Calculate overall
    response_scores = []
    for ev in all_evaluations:
        avg = sum(ev['response_eval'][m] for m in metrics) / len(metrics)
        response_scores.append(avg)

    tool_scores = []
    for ev in all_evaluations:
        avg = sum(ev['tool_eval'][m] for m in tool_metrics) / len(tool_metrics)
        tool_scores.append(avg)

    overall_response_avg = round(sum(response_scores) / len(response_scores), 2) if response_scores else 0
    overall_tool_avg = round(sum(tool_scores) / len(tool_scores), 2) if tool_scores else 0
    overall_score = round((overall_response_avg + overall_tool_avg) / 2, 2)

    # Strengths and weaknesses
    strengths = []
    weaknesses = []
    for m in metrics:
        avg_score = round(sum(agg[m]['scores']) / len(agg[m]['scores']), 2) if agg[m]['scores'] else 0
        if avg_score >= 7:
            strengths.append(f"{m}: {avg_score}/10")
        elif avg_score < 5:
            weaknesses.append(f"{m}: {avg_score}/10")

    for m in tool_metrics:
        avg_score = round(sum(agg[m]['scores']) / len(agg[m]['scores']), 2) if agg[m]['scores'] else 0
        if avg_score >= 7:
            strengths.append(f"{m}: {avg_score}/10")
        elif avg_score < 5:
            weaknesses.append(f"{m}: {avg_score}/10")

    # Recommendations
    recommendations = []
    min_metric = min(metrics, key=lambda m: sum(agg[m]['scores']) / len(agg[m]['scores']) if agg[m]['scores'] else 0)
    min_val = round(sum(agg[min_metric]['scores']) / len(agg[min_metric]['scores']), 2) if agg[min_metric]['scores'] else 0
    recommendations.append(f"Focus on improving {min_metric} (score: {min_val}/10). Review agent instructions to add more emphasis on this aspect.")

    if overall_tool_avg < 7:
        recommendations.append("Tool selection needs improvement. Ensure the agent consistently calls the most relevant tools for each query type.")

    if weaknesses:
        recommendations.append(f"Address weak areas: {', '.join(w.split(':')[0] for w in weaknesses)}")
    else:
        recommendations.append("Continue monitoring performance and consider adding more specialized tools for edge cases.")

    # Generate report
    now = datetime.now()
    timestamp_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    display_ts = now.strftime("%Y-%m-%d %H:%M:%S")

    report_lines = []
    report_lines.append("# EcoHome Energy Advisor - Evaluation Report")
    report_lines.append("")
    report_lines.append(f"**Generated**: {display_ts}")
    report_lines.append(f"**Tests Executed**: {len(test_results)}")
    report_lines.append(f"**Context**: San Francisco, CA")
    report_lines.append("")
    report_lines.append("---")
    report_lines.append("")
    report_lines.append("## Executive Summary")
    report_lines.append("")
    report_lines.append(f"| Metric | Score |")
    report_lines.append(f"|--------|-------|")
    report_lines.append(f"| **Overall Response Quality** | {overall_response_avg}/10 |")
    report_lines.append(f"| **Overall Tool Usage** | {overall_tool_avg}/10 |")
    report_lines.append(f"| **Combined Overall Score** | **{overall_score}/10** |")
    report_lines.append("")
    report_lines.append("### Response Quality Metrics")
    report_lines.append("")
    report_lines.append("| Dimension | Average | Min | Max |")
    report_lines.append("|-----------|---------|-----|-----|")
    for m in metrics:
        scores = agg[m]['scores']
        avg = round(sum(scores) / len(scores), 2) if scores else 0
        mn = min(scores) if scores else 0
        mx = max(scores) if scores else 0
        report_lines.append(f"| {m} | {avg}/10 | {mn} | {mx} |")

    report_lines.append("")
    report_lines.append("### Tool Usage Metrics")
    report_lines.append("")
    report_lines.append("| Dimension | Average | Min | Max |")
    report_lines.append("|-----------|---------|-----|-----|")
    for m in tool_metrics:
        scores = agg[m]['scores']
        avg = round(sum(scores) / len(scores), 2) if scores else 0
        mn = min(scores) if scores else 0
        mx = max(scores) if scores else 0
        report_lines.append(f"| {m} | {avg}/10 | {mn} | {mx} |")

    report_lines.append("")
    report_lines.append("---")
    report_lines.append("")
    report_lines.append("## Per-Test Results")
    report_lines.append("")

    for i, ev in enumerate(all_evaluations):
        report_lines.append(f"### Test {i+1}: {ev['test_id']} ({ev['category']})")
        report_lines.append("")
        report_lines.append(f"**Question**: {ev['question']}")
        report_lines.append("")
        r = ev['response_eval']
        t = ev['tool_eval']
        report_lines.append(f"- **ACCURACY**: {r['ACCURACY']}/10")
        report_lines.append(f"- **RELEVANCE**: {r['RELEVANCE']}/10")
        report_lines.append(f"- **COMPLETENESS**: {r['COMPLETENESS']}/10")
        report_lines.append(f"- **USEFULNESS**: {r['USEFULNESS']}/10")
        report_lines.append(f"- **Tool Appropriateness**: {t['Tool Appropriateness']}/10")
        report_lines.append(f"- **Tool Completeness**: {t['Tool Completeness']}/10")
        report_lines.append(f"- **Tools Used**: {', '.join(t['tools_used']) if t['tools_used'] else 'None'}")
        report_lines.append(f"- **Expected Tools**: {', '.join(t['expected_tools'])}")
        report_lines.append(f"- **Response Feedback**: {r['feedback']}")
        report_lines.append(f"- **Tool Feedback**: {t['feedback']}")
        report_lines.append("")

    report_lines.append("---")
    report_lines.append("")
    report_lines.append("## Strengths")
    report_lines.append("")
    if strengths:
        for s in strengths:
            report_lines.append(f"- ✅ **{s}**")
    else:
        report_lines.append("- No significant strengths identified yet.")

    report_lines.append("")
    report_lines.append("## Weaknesses")
    report_lines.append("")
    if weaknesses:
        for w in weaknesses:
            report_lines.append(f"- ⚠️ **{w}**")
    else:
        report_lines.append("- No significant weaknesses identified.")

    report_lines.append("")
    report_lines.append("## Recommendations for Improvement")
    report_lines.append("")
    for i, rec in enumerate(recommendations, 1):
        report_lines.append(f"{i}. {rec}")

    report_lines.append("")
    report_lines.append("---")
    report_lines.append("")
    report_lines.append("*Report generated by EcoHome Energy Advisor Evaluation Pipeline*")

    report_text = "\n".join(report_lines)

    reports_dir = Path("reports")
    reports_dir.mkdir(exist_ok=True)

    report_filename = f"evaluation_report_{timestamp_str}.md"
    report_path = reports_dir / report_filename
    report_path.write_text(report_text, encoding="utf-8")

    print("\n" + "=" * 60)
    print("EVALUATION REPORT SUMMARY")
    print("=" * 60)
    print(f"Report saved to: {report_path}")
    print(f"\nOverall Response Quality: {overall_response_avg}/10")
    print(f"Overall Tool Usage: {overall_tool_avg}/10")
    print(f"Combined Overall Score: {overall_score}/10")
    print(f"\nStrengths: {len(strengths)}")
    for s in strengths:
        print(f"  + {s}")
    print(f"\nWeaknesses: {len(weaknesses)}")
    for w in weaknesses:
        print(f"  - {w}")
    print(f"\nTop Recommendation: {recommendations[0]}")
    print("=" * 60)

    return {
        'overall_score': overall_score,
        'response_quality': overall_response_avg,
        'tool_usage': overall_tool_avg,
        'strengths': strengths,
        'weaknesses': weaknesses,
        'recommendations': recommendations,
        'report_path': str(report_path)
    }

In [14]:
# Run evaluation on all test results
eval_summary = generate_evaluation_report()


EVALUATION REPORT SUMMARY
Report saved to: reports\evaluation_report_2026-07-23_14-46-52.md

Overall Response Quality: 9.06/10
Overall Tool Usage: 8.12/10
Combined Overall Score: 8.59/10

Strengths: 6
  + ACCURACY: 9.25/10
  + RELEVANCE: 9.5/10
  + COMPLETENESS: 8.58/10
  + USEFULNESS: 8.92/10
  + Tool Appropriateness: 9.03/10
  + Tool Completeness: 7.23/10

Weaknesses: 0

Top Recommendation: Focus on improving COMPLETENESS (score: 8.58/10). Review agent instructions to add more emphasis on this aspect.
